# **11766 Advanced Perception for Mobile Robotics**
## Master's Degree in Intelligent Systems
### University of the Balearic Islands
---

##### Write in the following the names of the members of the group:
- John Smith 1
- John Smith 2

## **Instructions**

- **Do not delete any of the provided cells or functions**. Write your code where it is indicated. You may add new cells or functions as needed to complete your work.
- Along with this assignment, please submit a comprehensive **report** in PDF format explaining your implemented solutions. This report can include, for instance:
    - A technical explanation: Clearly describe the algorithms and techniques used, including relevant code snippets.
    - Results: Present your results in a clear and concise manner. Use visualizations such as graphs or tables to enhance understanding.
    - Analysis: Interpret your results and discuss their significance. Consider, for instance, the following questions:
        - Are the results as expected? Why or why not?
        - What factors might have influenced the results?
    - Conclusions: Summarize your findings and provide overall conclusions about your project.


# **Graph-based SLAM**

The goal of this assignment is to implement a least-squares based method to address the SLAM problem in its graph-based formulation as explained in the lectures.

We will consider 2D poses $(x, y, \theta)$ of the robot and 2D landmarks $(x_l, y_l)$ as the unknowns in our SLAM problem. The data is provided as a graph structure containing nodes (poses and landmarks) and constraints between these nodes (i.e pose-pose and pose-landmark). These datasets are stored as `g2o` text files. You are provided with the following datasets (see `data` folder), each of which represents the measurements of one SLAM problem:
1. `simulation-pose-pose.g2o`: simulated measurements containing pose-pose constraints only
2. `simulation-pose-landmark.g2o`: simulated measurements containing both pose-pose and pose-landmark constraints
3. `intel.g2o`: real world measurements containing pose-pose constraints only
4. `dlr.g2o`: real world measurements containing both pose-pose and pose-landmark constraints

To get started with this task, we provide some Python code which will help in loading the graph structure, visualize it and other functions that you may need. We also provide additional notes in the file `graph-slam-notes.pdf` which explains the expected results and the Jacobian computations in detail.

## **Understanding the Graph Structure**

Each graph consists of a set of nodes (or vertices) and edges that connect these nodes. As discussed in the lecture, the nodes correspond to the unknowns of the least-squares SLAM problem, whereas the edges correspond to the constraints obtained from the measurements. In this assignment, the graph has the following types of nodes and edges:

- Nodes:

    `VERTEX_SE2`: These nodes represent a 2D pose of the robot $(x, y, \theta)$

    `VERTEX_XY`: These nodes represent a 2D location of a landmark $(x_l, y_l)$
    
- Edges:

    `EDGE_SE2`: These edges represent a constraint between two VERTEX_SE2 nodes. We refer to these edges as pose-pose constraints.
    
    `EDGE_SE2_XY`: These edges represent a constraint between a VERTEX_SE2 node and a VERTEX_XY node. We refer these edges as pose-landmark edge.

In our code, we represent the graph as a class with the following attributes:
- `nodes`: A dictionary of nodes where the information of each `node` can be accessed with `nodeId` as a key. Each `node` has a unique `nodeId`. This node can be either `VERTEX_SE2` or `VERTEX_XY`. If node has a dimension of 3, it represents the pose the robot (`VERTEX_SE2`). If the node has a dimension of 2, it represents the location of landmark (`VERTEX_XY`).  
    
- `edges`: A list of all the `edges` in the graph where each `edge` has the following attributes:
    - `Type`: The type is 'P' if the constraint is a pose-pose constraint (`EDGE_SE2`), whereas it is 'L' if it is a pose-landmark constraint (`EDGE_SE2_XY`).
    - `fromNode`: `nodeId` of the node from which the edge originates from.
    - `toNode`: `nodeId` of the node to which the edge terminates to.
    - `measurement`: The measurement corresponding to the edge.
    - `information`: The corresponding information matrix for the edge constraint.
    
- `x`: All the unknowns (node variables) stacked into a vector. This should be used for updating the state after each iteration of the optimization.

- `lut`: This is a lookup table (implemented as a dictionary in Python). `lut[nodeId]` provides the starting location of the variables of the node with id `nodeId`.

Go through the examples in the cell below to understand how to work with the graph structure. Ensure that you understand how the graph is organized. This will be neccessary to solve all the tasks in the assignment. 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from utils import *
import numpy as np
%matplotlib inline

# load a dataset 
filename = 'data/simulation-pose-landmark.g2o'
graph = read_graph_g2o(filename)

# visualize the dataset
plot_graph(graph)
print('Loaded graph with {} nodes and {} edges'.format(len(graph.nodes), len(graph.edges)))

# print information for the two types of nodes
nodeId = 128
print('Node {} = {} is a VERTEX_SE2 node'.format(nodeId, graph.nodes[nodeId]))

# access the state vector using the lookup table
fromIdx = graph.lut[nodeId]
print('Node {} from the state vector = {}'.format(nodeId,graph.x[fromIdx:fromIdx+3]))

nodeId = 1
print('Node {} = {} is a VERTEX_XY node'.format(nodeId, graph.nodes[nodeId]))

# access the state vector using the lookup table
fromIdx = graph.lut[nodeId]
print('Node {} from the state vector = {}'.format(nodeId, graph.x[fromIdx:fromIdx+2]))

# print information for two types of edges
eid = 0 
print('Edge {} = {} is a pose-pose constraint'.format(eid, graph.edges[eid]))

eid = 1 
print('Edge {} = {} is a pose-landmark constraint'.format(eid, graph.edges[eid]))

## **Computing The Total Error**

**Task**:

Implement the function `compute_global_error` in the `utils.py` file. This functions computes the current error value for a graph with constraints. Compute this error for all the four datasets and print the results. You can see the results in the attached PDF file.

In [ ]:
# Compute the global error for all datasets

# YOUR CODE HERE
raise NotImplementedError()
# -----

## **Linearization of a Pose-Pose Constraint**

Since the constraint described by the edge between two pose nodes is non-linear, you need to linearize it solve the least-squares optimization. The function `linearize_pose_pose_constraint` computes the error and the Jacobian for a pose-pose constraint. It takes as input:

- $x1$: 3x1 vector\
    $x, y, \theta$ of the first robot pose
- $x2$: 3x1 vector\
    $x, y, \theta$ of the second robot pose
- $z$:  3x1 vector\
    $x, y,\theta$ of the measurement
    
and returns

- $e$: 3x1\
       Error of the constraint
- $A$: 3x3\
       Jacobian with regard to $x1$
- $B$: 3x3\
       Jacobian with regard to $x2$
         
**Task:**

Implement the `linearize_pose_pose_constraint` function in the `utils.py` file. Use the following cell to validate your Jacobian computation implementation by comparing them against numerically-computed Jacobians.

In [ ]:
import numpy as np

epsilon = 1e-5

x1 = np.array([1.1, 0.9, 1])
x2 = np.array([2.2, 1.85, 1.2])
z = np.array([0.9, 1.1, 1.05])

# Get the analytic Jacobian
e, A, B = linearize_pose_pose_constraint(x1, x2, z)

# Check the error vector
e_true = np.array([-1.06617, -1.18076, -0.85000])
if np.linalg.norm(e - e_true) > epsilon:
    print('Your error function seems to return a wrong value')
    print('Result of your function:')
    print(e)
    print('True value:')
    print(e_true)
else:
    print('The computation of the error vector appears to be correct')

# Compute it numerically
delta = 1e-6
scalar = 1 / (2 * delta)

# Test for x1
ANumeric = np.zeros((3, 3))
for d in range(3):
    curX = np.copy(x1)
    curX[d] += delta
    err = linearize_pose_pose_constraint(curX, x2, z)[0]
    curX = np.copy(x1)
    curX[d] -= delta
    err -= linearize_pose_pose_constraint(curX, x2, z)[0]

    ANumeric[:, d] = scalar * err

diff = ANumeric - A
if np.max(np.abs(diff)) > epsilon:
    print('Error in the Jacobian for x1')
    print('Your analytic Jacobian:')
    print(A)
    print('Numerically computed Jacobian:')
    print(ANumeric)
    print('Difference:')
    print(diff)
else:
    print('Jacobian for x1 appears to be correct')

# Test for x2
BNumeric = np.zeros((3, 3))
for d in range(3):
    curX = np.copy(x2)
    curX[d] += delta
    err = linearize_pose_pose_constraint(x1, curX, z)[0]
    curX = np.copy(x2)
    curX[d] -= delta
    err -= linearize_pose_pose_constraint(x1, curX, z)[0]

    BNumeric[:, d] = scalar * err

diff = BNumeric - B
if np.max(np.abs(diff)) > epsilon:
    print('Error in the Jacobian for x2')
    print('Your analytic Jacobian:')
    print(B)
    print('Numerically computed Jacobian:')
    print(BNumeric)
    print('Difference:')
    print(diff)
else:
    print('Jacobian for x2 appears to be correct')

## **Linearization of a Pose-Landmark Constraint**

Since the constraint described by the edge between a pose and landmark node is non-linear, you need to linearize it solve the least-squares optimization. The function `linearize_pose_landmark_constraint` computes the error and the Jacobian for pose-landmark constraint. It takes as input: 

- $x$: 3x1 vector\
       $x, y, \theta$ of the robot pose
- $l$: 2x1 vector\
       $x, y$ of the landmark
- $z$: 2x1 vector\
       $x,y$ of the measurement
    
and returns:

- $e$: 2x1 vector\
    Error for the constraint
- $A$: 2x3 vector\
    Jacobian with regard to $x$
- $B$: 2x2 vector\
    Jacobian with regard to $l$
    
**Task**:

Implement the `linearize_pose_landmark_constraint` function in the `utils.py` file. Use the following cell to validate your Jacobian computation implementation by comparing them against numerically-computed Jacobians.

In [ ]:
epsilon = 0.2

x1 = np.array([1.1, 0.9, 1])
x2 = np.array([2.2, 1.9])
z = np.array([1.3, -0.4])

# Get the analytic Jacobian
e, A, B = linearize_pose_landmark_constraint(x1, x2, z)

# Check the error vector
e_true = [0.135804, 0.014684]
if np.linalg.norm(e - np.array(e_true)) > epsilon:
    print('Your error function seems to return a wrong value')
    print('Result of your function:')
    print(e)
    print('True value:')
    print(e_true)
else:
    print('The computation of the error vector appears to be correct')

# Compute it numerically
delta = 1e-6
scalar = 1 / (2 * delta)

# Test for x1
ANumeric = np.zeros((2, 3))
for d in range(3):
    curX = x1.copy()
    curX[d] += delta
    err = np.array(linearize_pose_landmark_constraint(curX, x2, z)[0]).flatten()
    curX = x1.copy()
    curX[d] -= delta
    err -= np.array(linearize_pose_landmark_constraint(curX, x2, z)[0]).flatten()

    ANumeric[:, d] = scalar * err

diff = ANumeric - np.array(A)
if np.max(np.abs(diff)) > epsilon:
    print('Error in the Jacobian for x1')
    print('Your analytic Jacobian:')
    print(np.array(A))
    print('Numerically computed Jacobian:')
    print(ANumeric)
    print('Difference:')
    print(diff)
else:
    print('Jacobian for x1 appears to be correct')

# Test for x2
BNumeric = np.zeros((2, 2))
for d in range(2):
    curX = x2.copy()
    curX[d] += delta
    err = linearize_pose_landmark_constraint(x1, curX, z)[0].flatten()
    curX = x2.copy()
    curX[d] -= delta
    err -= linearize_pose_landmark_constraint(x1, curX, z)[0].flatten()

    BNumeric[:, d] = scalar * err

diff = BNumeric - np.array(B)
if np.max(np.abs(diff)) > epsilon:
    print('Error in the Jacobian for x2')
    print('Your analytic Jacobian:')
    print(np.array(B))
    print('Numerically computed Jacobian:')
    print(BNumeric)
    print('Difference:')
    print(diff)
else:
    print('Jacobian for x2 appears to be correct')

## **Building and Solving the Linearized System**

The `linearize_and_solve` function builds the $H$ and $b$ matrices in order to obtain $dx$ (i.e. change in the unknowns $x$) for one iteration. The function takes as input:

- $g$: graph at iteration $i$
    
and returns

- $dx$: Nx1 vector\
        Change in the solution for the unknowns $x$
**Task**:

Implement the `linearize_and_solve` function in the `utils.py` file. Some skeletal code for the function is already provided to you to start with.

In [ ]:
filename = 'data/simulation-pose-landmark.g2o'
g = read_graph_g2o(filename)
dx = linearize_and_solve(g)
print(f"For first iteration\n {dx}")

## **Iterative Procedure for Solving Non-Linear Least Squares**

The `run_graph_slam` function iteratively solves the least squares problem and updates the unknowns $x$. The procedure should be terminated if the change in $|dx| < 10^-4$ or the until a maximum number of iterations (maxIter = 100) is reached.

**Task**:

Implement the function `run_graph_slam` to perform the optimization. Some hints are provided as comments in the function. Test the function on the `simulation-pose-pose.g2o` and `simulation-pose-landmark.g2o` datasets.

## **Results for Different Datasets**

Evaluate the results of the Graph-SLAM algorithm for all the four datasets. For each one:

- Plot the graph before and after optimization
- Print the global error before and after optimization
- Plot the error vs iterations

In [ ]:
# Evaluate the four provided datasets

# YOUR CODE HERE
raise NotImplementedError()
# -----